# **Dataset Preprocessing: Edge-Preserving Smoothing**

In this work, the dataset images are preprocessed using **edge-preserving Gaussian smoothing**. This technique reduces high-frequency noise and small artifacts in the images while keeping important structural edges sharp. Specifically:  

1. **Edge Detection:** The Canny algorithm is used to identify prominent edges in grayscale versions of the images.  
2. **Edge Dilation:** Edges are dilated to ensure smoothing affects a small neighborhood around the contours.  
3. **Localized Gaussian Smoothing:** A Gaussian kernel is applied only along the detected edges, reducing noise without blurring critical structures.  
4. **Standardization:** All images are resized to a consistent resolution to ensure uniformity across the dataset.  

This preprocessing serves multiple purposes: it **removes irrelevant noise**, **preserves essential details** that define shapes and forms, and **stabilizes the training of GAN-based models**, allowing the network to focus on learning meaningful patterns rather than overfitting to image artifacts.

In [ ]:
# Importar librerías
import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
import cv2
import os
import numpy as np
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
import multiprocessing.dummy as mp

n_threads = 16
p = mp.Pool(n_threads)

In [ ]:
# Montar Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
def edge_job(args):
    output_size = 256, 256  # Final size

    # Unpack input arguments
    path, gauss, img_size, kernel, kernel_size, save, n = args

    # Read the original image)
    rgb_img = cv2.imread(path)

    # Read the grayscale version
    gray_img = cv2.imread(path, 0)

    # Check if the image was loaded correctly
    if rgb_img is None:
        print(path, "Error!")
        return

    # Resize the RGB image to img_size
    rgb_img = np.array(ImageOps.fit(Image.fromarray(rgb_img), img_size, Image.Resampling.LANCZOS))

    # Add 3-pixel reflected padding on all sides
    pad_img = np.pad(rgb_img, ((3, 3), (3, 3), (0, 0)), mode='reflect')

    # Resize the grayscale image to img_size
    gray_img = np.array(ImageOps.fit(Image.fromarray(gray_img), img_size, Image.Resampling.LANCZOS))

    # Detect edges using the Canny algorithm
    edges = cv2.Canny(gray_img, 150, 500)

    # Dilate the edges to make them thicker using the provided kernel
    dilation = cv2.dilate(edges, kernel)

    # Create a copy of the RGB image for Gaussian smoothing
    _gauss_img = np.copy(rgb_img)
    gauss_img = fast_loop(_gauss_img, pad_img, kernel_size, gauss, dilation)

    # Resize both the original and smoothed images to the final output size
    rgb_img = cv2.resize(rgb_img, output_size, Image.Resampling.LANCZOS)
    gauss_img = cv2.resize(gauss_img, output_size)

    # Combine the original and smoothed images side by side (horizontally)
    comb_img = np.concatenate((rgb_img, gauss_img), axis=1)

    # Save them
    cv2.imwrite(os.path.join(save, str(n) + '.jpg'), comb_img, [int(cv2.IMWRITE_JPEG_QUALITY), 95])

In [ ]:
def fast_loop(gauss_img, pad_img, kernel_size, gauss, dilation):
    # Get coordinates of edge pixels
    idx = np.where(dilation != 0)
    loops = int(np.sum(dilation != 0))

    # Apply the Gaussian kernel at each edge pixel
    for i in range(loops):
        gauss_img[idx[0][i], idx[1][i], 0] = np.sum(np.multiply(
            pad_img[idx[0][i]:idx[0][i] + kernel_size, idx[1][i]:idx[1][i] + kernel_size, 0], gauss))
        gauss_img[idx[0][i], idx[1][i], 1] = np.sum(np.multiply(
            pad_img[idx[0][i]:idx[0][i] + kernel_size, idx[1][i]:idx[1][i] + kernel_size, 1], gauss))
        gauss_img[idx[0][i], idx[1][i], 2] = np.sum(np.multiply(
            pad_img[idx[0][i]:idx[0][i] + kernel_size, idx[1][i]:idx[1][i] + kernel_size, 2], gauss))

    return gauss_img

In [ ]:
def edge_promoting(root, save):
    img_size = (384, 384)
    file_list = os.listdir(root)
    if not os.path.isdir(save):
        os.makedirs(save)

    # Kernel settings for edge detection & smoothing
    kernel_size = 5
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    gauss = cv2.getGaussianKernel(kernel_size, 0)
    gauss = gauss * gauss.transpose(1, 0)

    pbar = tqdm.tqdm(total=len(file_list))

    job_args = [(os.path.join(root, f), gauss, img_size, kernel, kernel_size, save, n) for n, f in enumerate(file_list)]

    # Run the edge_job function in parallel
    for _ in p.imap_unordered(edge_job, job_args):
        pbar.update(1)

In [ ]:
import zipfile

# Extract the images saved on our zip file
zip_path = "/content/drive/MyDrive/GAN/dataset_GAN.zip"
extract_path = "/content/dataset_GAN"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [ ]:
# Specify the folder with images you want to use for creating a smoothed dataset.
edge_promoting("/content/dataset_GAN/train/cartoon", "/content/dataset_GAN/train/cartoon_smoothed")

100%|██████████| 1544/1544 [06:32<00:00,  3.94it/s]


In [ ]:
#Save our new version of the dataset
import shutil

shutil.make_archive("/content/drive/MyDrive/GAN/dataset_GAN_smooth",'zip',"/content/dataset_GAN")
print("Zip saved on Drive")